In [1]:
from statsbombpy import sb
import pandas as pd

In [2]:
# 1. Obtener todas las competencias disponibles
competencias = sb.competitions()
resumen_temporadas = competencias['competition_name'].value_counts()
print(resumen_temporadas)

/Users/jzavaleta/Library/Python/3.14/lib/python/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


competition_name
Champions League           18
La Liga                    18
FIFA World Cup              8
FA Women's Super League     4
Copa del Rey                3
Ligue 1                     3
1. Bundesliga               2
Liga Profesional            2
NWSL                        2
Premier League              2
Serie A                     2
UEFA Euro                   2
UEFA Women's Euro           2
Women's World Cup           2
African Cup of Nations      1
Copa America                1
FIFA U20 World Cup          1
Frauen Bundesliga           1
Indian Super league         1
Liga F                      1
Major League Soccer         1
North American League       1
Serie A Women               1
UEFA Europa League          1
Name: count, dtype: int64


In [3]:
# 1. Creamos una copia para evitar advertencias
df_temp = competencias.copy()

# 2. Convertimos el inicio del año de la temporada a entero
df_temp['season_year'] = df_temp['season_name'].str[:4].astype(int)

# 3. Aplicamos el filtro
wc = df_temp[
    (df_temp['competition_name'] == "FIFA World Cup") & 
    (df_temp['season_year'] >= 2018)
]

print(wc)

    competition_id  season_id   country_name competition_name  \
30              43        106  International   FIFA World Cup   
31              43          3  International   FIFA World Cup   

   competition_gender  competition_youth  competition_international  \
30               male              False                       True   
31               male              False                       True   

   season_name               match_updated           match_updated_360  \
30        2022  2026-05-04T01:48:57.914346  2026-05-04T01:53:40.309717   
31        2018  2026-05-15T15:48:12.789440     2021-06-13T16:17:31.694   

           match_available_360             match_available  season_year  
30  2026-05-04T01:53:40.309717  2026-05-04T01:48:57.914346         2022  
31                         NaN  2026-05-15T15:48:12.789440         2018  


In [4]:
# 1. Obtenemos los partidos de ambas temporadas
wc_22 = sb.matches(competition_id=43, season_id=106)
wc_18 = sb.matches(competition_id=43, season_id=3)

# 2. Los concatenamos en un solo DataFrame
wc_totales = pd.concat([wc_22, wc_18], ignore_index=True)

# 3. Filtrar los partidos donde México jugó como local o visitante
partidos_mexico = wc_totales[
    (wc_totales['home_team'] == 'Mexico') | (wc_totales['away_team'] == 'Mexico')
].copy()

# 4. Asignamos la etiqueta 'wc_22' o 'wc_18' en el DataFrame de partidos según season_id
partidos_mexico['competencia'] = partidos_mexico['season_id'].map({106: 'wc_22', 3: 'wc_18'})

# 5. Descargamos los eventos iterando sobre los match_id y sus respectivas etiquetas de competencia
lista_eventos = []

for _, fila in partidos_mexico.iterrows():
    m_id = fila['match_id']
    comp_nombre = fila['competencia']
    
    # Descargamos los eventos del partido
    df_eventos = sb.events(match_id=m_id)
    
    # Añadimos la columna de la competencia
    df_eventos['competencia'] = comp_nombre
    
    # Guardamos en nuestra lista
    lista_eventos.append(df_eventos)

# 6. Concatenamos todos los DataFrames de eventos en uno solo
eventos_mexico_totales = pd.concat(lista_eventos, ignore_index=True)

print(f"\nTotal de eventos cargados de todos los partidos de México: {len(eventos_mexico_totales)}")

# Verificamos la distribución de eventos por competencia
print("\nDistribución de eventos por torneo:")
print(eventos_mexico_totales['competencia'].value_counts())

# Guardamos a CSV
eventos_mexico_totales.to_csv('wc_Mexico.csv', index=False, encoding='utf-8')

/Users/jzavaleta/Library/Python/3.14/lib/python/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/Users/jzavaleta/Library/Python/3.14/lib/python/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/Users/jzavaleta/Library/Python/3.14/lib/python/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/Users/jzavaleta/Library/Python/3.14/lib/python/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/Users/jzavaleta/Library/Python/3.14/lib/python/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/Users/jzavaleta/Library/Python/3.14/lib/python/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credenti


Total de eventos cargados de todos los partidos de México: 22686

Distribución de eventos por torneo:
competencia
wc_18    13215
wc_22     9471
Name: count, dtype: int64
